# Fine-tuning GRU для прогноза глюкозы

Модель предварительно обучается на виртуальных пациентах SimGlucose, затем адаптируется на ограниченной части персонального экспорта. Задача: прогноз CGM через 30 минут по 60 минутам истории.

> Исследовательская работа. Модель не рассчитывает дозу инсулина и не предназначена для лечебных решений.

In [ ]:
from pathlib import Path

EXPORT_ZIP = Path(r"C:\\path\\to\\glucotracker-export.zip")
OUTPUT_DIR = Path("results")
assert EXPORT_ZIP.exists(), "Укажите существующий ZIP-экспорт Glucotracker"


## Полный эксперимент

Скрипт читает CSV непосредственно из ZIP, исключает corrupted sensor sessions, генерирует синтетическую популяцию и сравнивает четыре метода.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "run_experiment.py", "--export-zip", str(EXPORT_ZIP), "--output-dir", str(OUTPUT_DIR)],
    check=True,
)


## Метрики

In [ ]:
import json
import pandas as pd

payload = json.loads((OUTPUT_DIR / "metrics.json").read_text(encoding="utf-8"))
pd.DataFrame(payload["metrics"]).T

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(OUTPUT_DIR / "comparison.png")))
display(Image(filename=str(OUTPUT_DIR / "training_curves.png")))

Заполненный текст отчёта находится в `results/LAB_REPORT.md`.

## Эксперимент с meal_insulin_episode_snapshots

В этом эксперименте модель прогнозирует глюкозу через 2 часа после пищевого эпизода. Значение `+2h` используется только как целевая переменная.

In [ ]:
subprocess.run(
    [sys.executable, "episode_experiment.py", "--export-zip", str(EXPORT_ZIP), "--output-dir", str(OUTPUT_DIR)],
    check=True,
)


In [ ]:
episode_payload = json.loads((OUTPUT_DIR / "episode_metrics.json").read_text(encoding="utf-8"))
display(pd.DataFrame(episode_payload["metrics"]).T)
display(Image(filename=str(OUTPUT_DIR / "episode_comparison_ru.png")))
display(Image(filename=str(OUTPUT_DIR / "episode_predictions_ru.png")))